## Colab to run Gemma3n model on CPU for a dataset manifest.

The google drive version of this colab (for iterating and editing) [is here](https://colab.research.google.com/drive/1YkS0FNqn-GeLALKVry5aTP25IrsSFWaf). Once you have a unit of edit completed, you can check-in the drive version into [this github directory](https://github.com/watch-duty/radio-transcription/tree/main/model/colabs).

In [1]:
#@title Imports and credentials setup

from transformers import pipeline
from pydub import AudioSegment
import torch
import json
from IPython.display import Audio, display

from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()  # This looks for the .env file
# This is the token for logging into hugging face, as a consent
# needs to be given to use gemma. You can get this by logging
# into hugging face and putting the login token into your
# user directory in a .env file.
HF_LOGIN_TOKEN = os.getenv("HF_LOGIN_TOKEN")
assert HF_LOGIN_TOKEN is not None, "HF_LOGIN_TOKEN not found in environment"
login(HF_LOGIN_TOKEN)

dataset_json = "/usr/local/google/home/varungulshan/radio-transcription/model/data/inference_manifests/playground_parakeet_and_canary_flash.json"  #@param
output_json = "/usr/local/google/home/varungulshan/radio-transcription/model/data/inference_manifests/playground_parakeet_and_canary_flash_and_gemma3n_e2b_it.json"  #@param

In [2]:
#@title Load the model
pipe = pipeline(
    "image-text-to-text",
    # TODO: Experiment with the variations here.
    model="google/gemma-3n-e2b-it",
    device="cpu",
    # Using float32 right now because the audio input seems to come in
    # float32 -- if we figure out how to use float16 for transforming
    # the input, we could probably save lot of memory.
    torch_dtype=torch.float32,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


In [3]:
#@title Helper functions

def run_gemma_on_dataset(dataset_json, output_json):
  # This is used to save the snipped audio temporarily.
  # Only works with a single threaded process and no other
  # process using the same filename, so beware!
  temp_audio_filename = "/tmp/audio.wav"

  with open(dataset_json, 'r', encoding='utf-8') as f:
    with open(output_json, 'w', encoding='utf-8') as out:
      # Iterate through each utterance output in the json
      for line in f:
        data_row = json.loads(line)
        audio_filepath = data_row["audio_filepath"]

        # Convert the audio to a .wav file and snip it to the segment
        # specified in the json.
        audio = AudioSegment.from_file(audio_filepath)
        start_ms = int(data_row["offset"] * 1000)
        end_ms = int((data_row["offset"] + data_row["duration"]) * 1000)
        # TODO: Do we need to assert 30 second max for the audio? review gemma documentation.
        audio = audio[start_ms:end_ms]
        audio.export(temp_audio_filename, format="wav")

        messages = [
            {
                "role": "system",
                # TODO: Do we need this?
                "content": [{"type": "text", "text": "You are a helpful assistant."}]
            },
            # TODO: Can specify language is English in the prompt?
            # The instructions to output an empty string don't work right now.
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": temp_audio_filename},
                    {"type": "text", "text":
                     ("Transcribe this audio for me please. " +
                      "Do not add any other text except for the transcription. " +
                      "If you cannot transcribe, then just output an empty string."
                     )}
                ]
            }
        ]

        print(f"Transcribing {audio_filepath}:{start_ms}-{end_ms} with gemma.")
        # TODO: Review this max tokens thing
        # TODO: Gemma is not deterministic right now, how to configure that?
        output = pipe(text=messages, max_new_tokens=200)
        transcription = output[0]["generated_text"][-1]["content"]
        print(transcription)
        data_row["pred_text_gemma3n"] = transcription
        out.write(json.dumps(data_row) + "\n")
        # display(Audio(temp_audio_filename, autoplay=False))
        # _ignore = input("Press enter to continue to next item")

In [ ]:
run_gemma_on_dataset(dataset_json, output_json)

Transcribing /usr/local/google/home/varungulshan/watchduty-asr/data_exports/playground_export/5d995a78-ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15.mp3:2570-8507 with gemma.
I'm sorry, I can't transcribe that audio. It appears to be a repetition of the word "Run" and doesn't contain any discernible speech.
Transcribing /usr/local/google/home/varungulshan/watchduty-asr/data_exports/playground_export/5d995a78-ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15.mp3:9322-13301 with gemma.
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R
R

Transcribing /usr/local/google/home/varungulshan/watchduty-asr/data_exports/playground_export/5d995a78-ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15.mp3:13320-18336 with gemma.
I'm done in point three and ten. You love clear and can do that.
Transcribing /usr/local/google/home/varungulshan/watchduty-asr/